## Import modules

In [1]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scikitplot as skplt

from yellowbrick.model_selection import LearningCurve, ValidationCurve
from scipy.stats import friedmanchisquare, wilcoxon

pd.set_option('display.max_columns', None)

## Master Function to scale data, split into train/test sets, and run each classifier

In [2]:
partitions = [
    ("20/80", 0.80),  # 20% train, 80% test
    ("50/50", 0.50),  # 50% train, 50% test
    ("80/20", 0.20)   # 80% train, 20% test
]

def Scalar(trd, ted, cols):
    trd = trd.copy()
    ted = ted.copy()
    if not cols:
        return trd, ted, None
    scaler = StandardScaler()
    trd[cols] = scaler.fit_transform(trd[cols])
    ted[cols] = scaler.transform(ted[cols])
    return trd, ted, scaler

def master_function(X, y, dataset, classifier, estimator, param_grid, numeric_cols, results_list):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=9)

    seeds = [1, 2, 3]

    for trial, seed in enumerate(seeds, start=1):
        for partition, test_size in partitions:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=test_size, random_state=seed, stratify=y
            )
            X_train_scaled, X_test_scaled, scaler = Scalar(X_train, X_test, numeric_cols)
            train_size = len(y_train)
            test_size_n = len(y_test)

            grid = GridSearchCV(
                estimator=estimator,
                param_grid=param_grid,
                cv=cv,
                scoring="accuracy",
                n_jobs=-1,
                refit=True,
                return_train_score=True
            )

            grid.fit(X_train_scaled, y_train)

            mean_train_curve = grid.cv_results_["mean_train_score"]
            std_train_curve  = grid.cv_results_["std_train_score"]
            mean_val_curve   = grid.cv_results_["mean_test_score"]
            std_val_curve    = grid.cv_results_["std_test_score"]
            param_settings = grid.cv_results_["params"]


            best_model = grid.best_estimator_

            y_train_pred = best_model.predict(X_train_scaled)
            y_test_pred = best_model.predict(X_test_scaled)

            mean_scores = grid.cv_results_["mean_test_score"]
            std_scores = grid.cv_results_["std_test_score"]
            best_index = grid.best_index_

            train_acc = accuracy_score(y_train, y_train_pred)
            val_acc = float(mean_scores[best_index])
            val_std = float(std_scores[best_index])
            test_acc = accuracy_score(y_test, y_test_pred)

            precision = precision_score(y_test, y_test_pred, zero_division=0)
            recall = recall_score(y_test, y_test_pred, zero_division=0)
            f1 = f1_score(y_test, y_test_pred)


            results_list.append({
                "dataset": dataset,
                "partition": partition,
                "trial": trial,
                "train_size": train_size,
                "test_size": test_size_n,
                "classifier": classifier,
                "train_acc": train_acc,
                "val_acc": val_acc,
                "val_std": val_std,
                "test_acc": test_acc,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "best_params": grid.best_params_,
                "train_curve": mean_train_curve,
                "val_curve": mean_val_curve,
                "params_curve": param_settings,

            })


## Hyperparameters

In [3]:
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 11, 15],
    "weights": ["uniform", "distance"],
    "metric": ["minkowski"]
}

param_grid_svc = {
    "C": [0.1, 1, 10],
    "gamma": ["scale", 0.01, 0.001],
    "kernel": ["rbf"]
}

param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5]
}

param_grid_logreg = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs"],
    "max_iter": [1000]
}

param_grid_ada = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1.0]
}

## Classifiers

In [4]:
classifiers = [
    ("KNN", KNeighborsClassifier(), param_grid_knn),
    ("SVM-RBF", SVC(), param_grid_svc),
    ("RandomForest", RandomForestClassifier(random_state=9), param_grid_rf),
    ("LogisticRegression", LogisticRegression(), param_grid_logreg),
    ("AdaBoost", AdaBoostClassifier(random_state=9), param_grid_ada)                                                                                                                                                    
]

## Adult Dataset 

In [5]:
adult_columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
adult = pd.read_csv('adult.data', names = adult_columns, header=None, skipinitialspace=True )

In [6]:
adult = adult.replace("?", np.nan)
adult = adult.dropna()

In [7]:
A_categorical_columns = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
adult = pd.get_dummies(adult, columns=A_categorical_columns, drop_first=True)

In [8]:
adult['target'] = (adult['income'] == '>50K').astype(int)
adult = adult.drop(columns = ['income'])

In [9]:
X1 = adult.drop(columns=['target'])
y1 = adult['target']

In [10]:
A_numeric_columns = ['age', 'fnlwgt', 'education-num',
            'capital-gain', 'capital-loss', 'hours-per-week']

In [ ]:
results_adult = []

for clf_name, estimator, param_grid in classifiers:
    master_function(
        X=X1,
        y=y1,
        dataset="Adult",
        classifier=clf_name,
        estimator=estimator,
        param_grid=param_grid,
        numeric_cols=A_numeric_columns,
        results_list=results_adult
    )

df_adult = pd.DataFrame(results_adult)
print(df_adult)

summary_adult = (
    df_adult
    .groupby(["classifier", "partition"])
    [["test_acc", "precision", "recall", "f1"]]
    .agg(["mean", "std"])
)

summary_adult.columns = [
    'Accuracy (mean)', 'Accuracy (std)',
    'Precision (mean)', 'Precision (std)',
    'Recall (mean)', 'Recall (std)',
    'F1-score (mean)', 'F1-score (std)'
]

print(summary_adult)


   dataset partition  trial  train_size  test_size          classifier  \
0    Adult     20/80      1        6032      24130                 KNN   
1    Adult     50/50      1       15081      15081                 KNN   
2    Adult     80/20      1       24129       6033                 KNN   
3    Adult     20/80      2        6032      24130                 KNN   
4    Adult     50/50      2       15081      15081                 KNN   
5    Adult     80/20      2       24129       6033                 KNN   
6    Adult     20/80      3        6032      24130                 KNN   
7    Adult     50/50      3       15081      15081                 KNN   
8    Adult     80/20      3       24129       6033                 KNN   
9    Adult     20/80      1        6032      24130             SVM-RBF   
10   Adult     50/50      1       15081      15081             SVM-RBF   
11   Adult     80/20      1       24129       6033             SVM-RBF   
12   Adult     20/80      2        603

## Cover Type Dataset

In [13]:
cover_type_columns = [
    'Elevation',
    'Aspect',
    'Slope',
    'Horizontal_Distance_To_Hydrology',
    'Vertical_Distance_To_Hydrology',
    'Horizontal_Distance_To_Roadways',
    'Hillshade_9am',
    'Hillshade_Noon',
    'Hillshade_3pm',
    'Horizontal_Distance_To_Fire_Points',
    'Wilderness_Area1',
    'Wilderness_Area2',
    'Wilderness_Area3',
    'Wilderness_Area4',
] + [f'Soil_Type{i}' for i in range(1, 41)] + [
    'Cover_Type'
]

cover_type = pd.read_csv('covtype.data.gz', header=None, names=cover_type_columns)

In [14]:
cover_type['target'] = (cover_type['Cover_Type'] == 2).astype(int)
cover_type = cover_type.drop(columns='Cover_Type')

In [15]:
X2 = cover_type.drop(columns = ['target'])
y2 = cover_type['target']

In [16]:
cv_numeric_columns = ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points']

In [ ]:
results_cover_type = []

for clf_name, estimator, param_grid in classifiers:
    master_function(
        X=X2,
        y=y2,
        dataset="Cover Type",
        classifier=clf_name,
        estimator=estimator,
        param_grid=param_grid,
        numeric_cols=cv_numeric_columns,
        results_list=results_cover_type
    )

df_cover_type = pd.DataFrame(results_cover_type)
print(df_cover_type)

summary_cover_type = (
    df_cover_type
    .groupby(["classifier", "partition"])
    [["test_acc", "precision", "recall", "f1"]]
    .agg(["mean", "std"])
)

summary_cover_type.columns = [
    'Accuracy (mean)', 'Accuracy (std)',
    'Precision (mean)', 'Precision (std)',
    'Recall (mean)', 'Recall (std)',git addn 
    'F1-score (mean)', 'F1-score (std)'
]

print(summary_cover_type)


## Letter Dataset

In [ ]:
letter_columns = ['lettr', 'x-box', 'y-box', 'width','high', 'onpix', 'x-bar', 'y-bar', 'x2bar', 'y2bar', 'xybar', 'x2ybr', 'xy2br', 'x-ege', 'xegvy', 'y-ege', 'yegvx']
letter = pd.read_csv('letter-recognition.data', header=None, names=letter_columns)

letter['target'] = (letter['lettr'] == 'O').astype(int)
letter = letter.drop(columns=['lettr'])

In [ ]:
X3 = letter.drop(columns = ['target'])
y3 = letter['target']

In [ ]:
ltr_numeric_columns = X3.columns

In [ ]:
results_letter = []

for clf_name, estimator, param_grid in classifiers:
    master_function(
        X=X3,
        y=y3,
        dataset="Letter",
        classifier=clf_name,
        estimator=estimator,
        param_grid=param_grid,
        numeric_cols=ltr_numeric_columns,
        results_list=results_letter
    )

df_letter = pd.DataFrame(results_letter)
print(df_letter)

summary_letter = (
    df_letter
    .groupby(["classifier", "partition"])
    [["test_acc", "precision", "recall", "f1"]]
    .agg(["mean", "std"])
)
print(summary_letter)


## House Votes Dataset

In [ ]:
hv_columns = [
    'Class',
    'handicapped-infants',
    'water-project-cost-sharing',
    'adoption-of-the-budget-resolution',
    'physician-fee-freeze',
    'el-salvador-aid',
    'religious-groups-in-schools',
    'anti-satellite-test-ban',
    'aid-to-nicaraguan-contras',
    'mx-missile',
    'immigration',
    'synfuels-corporation-cutback',
    'education-spending',
    'superfund-right-to-sue',
    'crime',
    'duty-free-exports',
    'export-administration-act-south-africa'
]

house_votes = pd.read_csv('house-votes-84.data', header=None, names=hv_columns)

In [ ]:
house_votes.eq('?').sum()

In [ ]:
house_votes = house_votes.replace('?', 'unknown')

In [ ]:
house_votes['target'] = (house_votes['Class'] == 'republican').astype(int)
house_votes = house_votes.drop(columns = ['Class'])

In [ ]:
hv_cat_cols = house_votes.drop(columns = ['target']).columns
house_votes = pd.get_dummies(house_votes, columns=hv_cat_cols, drop_first=True)

X4 = house_votes.drop(columns = ['target'])
y4 = house_votes['target']

In [ ]:
results_House_votes = []

for clf_name, estimator, param_grid in classifiers:
    master_function(
        X=X4,
        y=y4,
        dataset="House Votes",
        classifier=clf_name,
        estimator=estimator,
        param_grid=param_grid,
        numeric_cols=[],
        results_list=results_House_votes
    )

df_House_votes = pd.DataFrame(results_House_votes)
print(df_House_votes)

summary_House_votes = (
    df_House_votes
    .groupby(["classifier", "partition"])
    [["test_acc", "precision", "recall", "f1"]]
    .agg(["mean", "std"])
)
print(summary_House_votes)


## Heart Disease Dataset

In [ ]:
hd_columns = heart_columns = [
    'age',
    'sex',
    'cp',
    'trestbps',
    'chol',
    'fbs',
    'restecg',
    'thalach',
    'exang',
    'oldpeak',
    'slope',
    'ca',
    'thal',
    'num'
]

heart_disease = pd.read_csv('processed.cleveland.data', header=None, names=hd_columns)

In [ ]:
heart_disease['target'] = (heart_disease['num'] > 0).astype(int)
heart_disease = heart_disease.drop(columns = ['num'])

In [ ]:
heart_disease = heart_disease[~heart_disease.isin(['?']).any(axis=1)]
heart_disease = heart_disease.apply(pd.to_numeric)

In [ ]:
X5 = heart_disease.drop(columns = ['target'])
y5 = heart_disease['target']

In [ ]:
hd_numeric_columns = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

In [ ]:
results_Heart_disease = []

for clf_name, estimator, param_grid in classifiers:
    master_function(
        X=X5,
        y=y5,
        dataset="Heart Disease",
        classifier=clf_name,
        estimator=estimator,
        param_grid=param_grid,
        numeric_cols=hd_numeric_columns,
        results_list=results_Heart_disease
    )

df_Heart_disease = pd.DataFrame(results_Heart_disease)
print(df_Heart_disease)

summary_Heart_disease = (
    df_Heart_disease
    .groupby(["classifier", "partition"])
    [["test_acc", "precision", "recall", "f1"]]
    .agg(["mean", "std"])
)
print(summary_Heart_disease)
